# 02 · Urban data formats and sources

*Notebooks: [01 Overview](01-overview.ipynb) · [02 Formats](02-formats.ipynb) · [03 Data](03-data.ipynb) · [04 Maps](04-maps.ipynb) · [05 Plots](05-plots.ipynb) · [06 Compute](06-compute.ipynb)*

Urban data comes as **vector** data (points, lines and polygons with attributes) or **raster** data (a grid of
values). This notebook shows the formats pyautark reads and the Python tools that produce them.

In [ ]:
import autark as ak
import tutorial as t

## 1 Vector geometry

A *feature* is a geometry plus properties. GeoJSON has seven geometry types: `Point`, `MultiPoint`, `LineString`,
`MultiLineString`, `Polygon`, `MultiPolygon` and `GeometryCollection`. The file below has one of each of the first,
third and fifth, in one collection.

In [ ]:
example = t.read_json("data/formats/example.geojson")
for feature in example["features"]:
    print(feature["geometry"]["type"].ljust(12), feature["properties"])

A source with inline `values` carries its data inside the spec. That is the most portable choice: it works in a
notebook, in Colab, in an exported HTML page. Autark decides how to draw a table from its geometry (`points`,
`polylines` or `polygons`), so the three geometry types below become three sources; a `Layer` does not say.

In [ ]:
# Autark draws a table in one way, from its geometry, so a collection that mixes types goes in as one source per type.
def collection(*features):
    return {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"id": i}, "geometry": g} for i, g in enumerate(features)]}

# The same three shapes as the file above, placed in lower Manhattan so they are visible at city scale.
polygons = ak.GeoJSON("polygons", values=collection(
    {"type": "Polygon", "coordinates": [[[-74.0125, 40.7035], [-74.0035, 40.7035], [-74.0035, 40.7095], [-74.0125, 40.7095], [-74.0125, 40.7035]]]}))
lines = ak.GeoJSON("lines", values=collection(
    {"type": "LineString", "coordinates": [[-74.0110, 40.7060], [-74.0090, 40.7075], [-74.0070, 40.7060], [-74.0050, 40.7075]]}))
points = ak.GeoJSON("points", values=collection({"type": "Point", "coordinates": [-74.0110, 40.7075]}))

# The first source sets the extent of the workspace, and later sources are cut to it. Here the polygon covers the
# other two, so it goes first.
ak.Spec(
    data=[polygons, lines, points],
    views=[ak.Map(layers=[
        ak.Layer(polygons).style(color="#4c78a8", opacity=0.5),
        ak.Layer(lines).style(color="#54a24b", width=20),
        ak.Layer(points).style(color="#e45756", size=80),
    ])],
).widget()

## 2 Formats

### 2.1 GeoJSON

`ak.GeoJSON(name, url=...)` or `ak.GeoJSON(name, values=...)`. GeoJSON coordinates are WGS84 longitude and latitude
(EPSG:4326), which is the default. If your file uses another system, say so with `coordinate_format="EPSG:3857"`: a
projected file read as degrees fails, with the extent of the layer in the message.

In [ ]:
neighborhoods = ak.GeoJSON("neighborhoods", url=t.url("data/map/mnt_neighs.geojson"))
neighborhoods.to_dict()

### 2.2 Well-known text

WKT writes a geometry as text, `POLYGON ((100 0, 101 0, 101 1, 100 1, 100 0))`. It usually lives in a column of a CSV
or JSON table. Shapely reads and writes it, so you can check it in Python first:

In [ ]:
from shapely import wkt

print(open(t.ROOT / "data/formats/example.wkt").read())
polygon = wkt.loads(open(t.ROOT / "data/formats/example.wkt").read())
print(polygon.geom_type, polygon.area, polygon.bounds)

A table with a WKT column is a `CSV` (or `JSON`) with `geometry=ak.wkt(column)`. The geometry type of each row comes
from the text.

In [ ]:
island = ak.CSV("island", url=t.url("data/db/mnt.csv"), geometry=ak.wkt("wkt"))

ak.Spec(
    data=[island],
    views=[ak.Map(layers=[ak.Layer(island).style(color="#72b7b2", opacity=0.7)])],
).widget()

A CSV with latitude and longitude columns uses `ak.latlng(latitude_column, longitude_column)` instead. The noise
complaints of notebook 04 are such a file:

In [ ]:
noise = ak.CSV("noise", url=t.url("data/map/mnt_noise.csv"),
               geometry=ak.latlng("latitude", "longitude"))

ak.Spec(
    data=[noise],
    views=[ak.Map(layers=[ak.Layer(noise).style(color="#f58518", opacity=0.6)])],
).widget()

### 2.3 GeoTIFF

A GeoTIFF is a raster: a grid of cells, each with a value, placed on the Earth by metadata. `ak.GeoTIFF(name, url=...)`
loads one as a raster table. Autark reads the file in the browser. Large rasters are downsampled to fit
`max_pixels` (default one million cells), and `resample_method` picks `"bilinear"` or `"nearest"`.

Autark does not frame the map from a raster, so the spec below also draws a vector layer over the same area. The
`Layer` for a raster is colored by value; a color scheme gives the ramp.

In [ ]:
elevation = ak.GeoTIFF("elevation", url=t.url("data/db/elevation.tif"))

# The raster covers lower Manhattan: longitude -74.018 to -73.971, latitude 40.700 to 40.725.
frame = ak.GeoJSON("frame", values={"type": "FeatureCollection", "features": [{
    "type": "Feature", "properties": {},
    "geometry": {"type": "Polygon", "coordinates": [[[-74.018, 40.700], [-73.971, 40.700], [-73.971, 40.725], [-74.018, 40.725], [-74.018, 40.700]]]},
}]})

ak.Spec(
    data=[frame, elevation],
    views=[ak.Map(layers=[
        ak.Layer(frame).style(color="#dddddd"),
        ak.Layer(elevation, id="elevation").encode(color=ak.field("value", ak.Scale(scheme="viridis"))).style(opacity=0.8),
    ])],
).widget()

### 2.4 OpenStreetMap

OpenStreetMap (OSM) is the open map of the world. It has no fixed schema: ways and relations carry free tags. Autark's
`OSM` source reads it into one table per layer: `surface`, `parks`, `water`, `roads` and `buildings`, named
`<name>_<layer>`.

An area comes from a place name (geocoded, queried from the Overpass API) or from a box `bbox=(west, south, east,
north)` in degrees. With `source="pbf"` and a `pbf_file_url`, the data is read from a local extract instead of the
network; this notebook folder has one for lower Manhattan.

In [ ]:
osm = ak.OSM(
    "fidi",
    bbox=(-74.02, 40.700, -73.995, 40.72),
    layers=["surface", "water", "parks", "roads", "buildings"],
    source="pbf",
    pbf_file_url=t.url("data/osm/lower_mnt.osm.pbf"),
)
print([osm.layer_source(layer) for layer in osm.layers])

ak.Spec(
    data=[osm],
    views=[ak.Map(layers=[ak.Layer(osm.layer_source(layer), id=layer) for layer in osm.layers])],
).widget()

The recorded file `data/formats/example.osm` shows the raw format: OSM XML, with nodes, ways and relations.
Autark's loader does that parsing for you; here it is with the standard library.

In [ ]:
import xml.etree.ElementTree as ET

root = ET.parse(t.ROOT / "data/formats/example.osm").getroot()
for element in root:
    print(element.tag, dict(element.attrib), [(c.tag, c.attrib) for c in element][:3])

## 3 From pandas and GeoPandas

Data you have in Python goes into a spec as inline values. `GeoJSON.from_dataframe` turns a table with latitude and
longitude columns into points; `GeoJSON.from_geopandas` takes a GeoDataFrame (reprojecting to EPSG:4326).

In [ ]:
import pandas as pd

trees = pd.DataFrame({
    "species": ["oak", "maple", "elm", "oak"],
    "lat": [40.7580, 40.7527, 40.7484, 40.7614],
    "lon": [-73.9855, -73.9772, -73.9857, -73.9776],
})
source = ak.GeoJSON.from_dataframe("trees", trees, latitude="lat", longitude="lon")

ak.Spec(
    data=[source],
    views=[ak.Map(layers=[ak.Layer(source).style(color="#54a24b")])],
).widget()

## 4 Open data portals

Cities publish data on open portals (for example NYC Open Data, data.gov and many municipal sites), usually as CSV or
GeoJSON. Download the file and point a source at it, or read it with pandas and pass it in as above. Mind the
coordinate system, and check the geometry type before you pick how to draw it.